# Experiments on $SynTab$ Dataset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt


# Parameters
n_classes = 5
n_features = 10
n_redundant = 2  # Number of redundant features
n_samples_per_class = 1000
random_state = 42

# Load the dataset
df = pd.read_csv("synthetic_dataset.csv")

# Separate features (X) and target labels (y)
X = df.drop(columns=['Class']).values
y = df['Class'].values

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# Standardize the features (important for neural networks)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression

# Define the Logistic Regression model with a linear decision boundary
model = LogisticRegression(max_iter=1000, multi_class='ovr', solver='lbfgs', random_state=42)

# Train the model
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
# Calculate the accuracy
test_acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
def sigmoid(x):
    s = 1 /(1 + np.exp(-x))
    return s

## get the indices for the reward function

In [ ]:
from copy import deepcopy

## Find the examples that uses redundent features for classes
counter = 0
counter1 = 0
index_list = []
X4test = deepcopy(X_test[1000:])
y4test = deepcopy(y_test[1000:])
expl = []
for i, (test, y) in enumerate(zip(X_test[:1000], y_test[:1000])):
    activation = test[np.newaxis, :] * model.coef_.T[:,int(model.predict(test[np.newaxis, :]))]
    expl.append(activation.squeeze())
    importance = np.argsort(activation).squeeze()[::-1]
    y_pred = model.predict(test[np.newaxis, :]).squeeze()
    if y_pred != y:
        index_list.append(i)
    if 8 in importance[:5] or 9 in importance[:5]:
        index_list.append(i)
target_list = list(set(index_list))


## Set for the Test Env

In [ ]:
# Predict on the test set
y4test_pred = model.predict(X4test)
y4test_proba = model.predict_proba(X4test)
# Calculate the accuracy
test_acc = accuracy_score(y4test, y4test_pred)
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
index4test_list = []
expl_test = []
index4test_list_expl = []
count = 0
for i, (test, y) in enumerate(zip(X4test, y4test)):
    activation = test[np.newaxis, :] * model.coef_.T[:,int(model.predict(test[np.newaxis, :]))]
    expl_test.append(activation.squeeze())
    importance = np.argsort(activation).squeeze()[::-1]
    y_pred = model.predict(test[np.newaxis, :]).squeeze()
    if y_pred != y:
        index4test_list.append(i)
    if 8 in importance[:5] or 9 in importance[:5]:
        if not i in index4test_list:
            index4test_list_expl.append(i)
            count += 1
        index4test_list.append(i)
target4test_list = list(set(index4test_list))

# RL implementation


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Optional
from copy import deepcopy

class AuditingEnv(gym.Env):
    """
    Custom Environment where the state is a matrix, and the action is selecting an index.
    """
    def __init__(self, state_shape=(25,), expl_shape=(50), action_list_length=50, target_list=[], dataset=[], explanation=[], dataset_label=[], max_steps=10000, test_env=False, index4test_list_expl=None):
        super(AuditingEnv, self).__init__()
        
        # Define the matrix as the observation space
        self.state_shape = state_shape
        self.expl_shape = expl_shape
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(state_shape[0] + expl_shape[0], )
        )
        self.target_list = deepcopy(target_list)
        self.dataset = np.float32(dataset)
        self.dataset_expl = np.float32(explanation)
        self.dataset_label = dataset_label

        # Define the action space as selecting an index from a list of length X
        self.action_list_length = action_list_length
        self.action_space = spaces.Discrete(action_list_length, start=0)

        # Initialize the state (the matrix)
        self.state = np.random.uniform(-0, 1, size=self.state_shape) #np.zeros(state_shape)
        self.current_step = 0
        self.max_steps = max_steps  # Maximum steps in an episode
        
        self.rewards = None
        self.episode_rewards = []
        
        self.target_unseen = []
        
        self.test_env = test_env
        
        self.diversity = np.ones(action_list_length)
        
        self.index4test_list_expl = index4test_list_expl

    def reset(self, init_state=None):
        """Resets the environment to the initial state."""
        self.rewards = []
        self.rewards_expl = []
        if not init_state is None:
            self.state = init_state[:self.state_shape[0]]  #np.random.uniform(-0, 1, size=self.state_shape)
            self.expl = init_state[self.state_shape[0]:]
        else:
            self.state = np.zeros(self.state_shape) #np.zeros(state_shape)
            self.expl = np.zeros(self.expl_shape)
        self.current_step = 0
        self.target_unseen = deepcopy(self.target_list)
        self.dataset_seen = []
        self.done = False

        self.diversity = np.ones(self.action_list_length) 
        state = np.concatenate((self.state, self.expl))
        return state, self.diversity

    def step(self, action):
        """
        Execute the action and return the next state, reward, done, and info.
        Args:
            action (int): Index of the target list chosen by the agent.
        Returns:
            observation (array): The matrix (state).
            reward (float): Reward for the action.
            done (bool): Whether the episode is finished.
            info (dict): Additional debug information.
        """
        action = int(action)
        self.current_step += 1

                
        # Check if action is valid
        assert self.action_space.contains(action), f"Invalid action: {action}, Action space: {self.action_space}"

        if not self.test_env:
            if action in self.target_unseen:
            # delete this sample
                self.target_unseen.remove(action)
                reward = 1 #+ entropy(self.diversity, base=base, axis=0)*0.1
            elif action in self.target_list:
                reward = 1 #+ entropy(self.diversity, base=base, axis=0)*0.1
            else:
                reward = 0
        else:
            if action in self.target_unseen:
                reward = 1
                self.target_unseen.remove(action)
                if not self.index4test_list_expl is None:
                    if action in self.index4test_list_expl:
                        reward_expl = 1
                    else:
                        reward_expl = 0
            else:
                reward = 0
                reward_expl = 0
            self.rewards_expl.append(reward_expl)

        if not action in self.dataset_seen:
            self.dataset_seen.append(action)

        # Update the states
        if action in self.target_list:
            self.diversity[action] += 1

        pred = np.argmax(self.dataset[action,:])

        self.state[pred*5:(pred+1)*5] += self.dataset[action,:]
        self.expl[pred*10:(pred+1)*10] += self.dataset_expl[action,:]        
        norm_rows = np.linalg.norm(self.state)
        norm_diver = np.linalg.norm(self.diversity)
        norm_expl_rows = np.linalg.norm(self.expl)
        # Check if the episode is done
        self.done = (len(self.target_unseen) == 0) or (self.current_step >= self.max_steps)

        # Debugging information
        info = self._get_info()
        
        self.rewards.append(reward)

        if self.done:
            ep_rew = sum(self.rewards)
            self.episode_rewards.append(ep_rew)

        state = np.concatenate((self.state/norm_rows, self.expl/norm_expl_rows))

        return state, reward, self.done, self.dataset_seen, self.diversity/norm_diver  #abs(arr-self.state/norm_rows)

    def _get_info(self):
           return {
             "Step": self.current_step,
             "Matrix": self.state,
             "Target List": self.target_list,
           "Reward": self.rewards,
           "Unseen reward": self.target_unseen,
           "Diversity list": self.diversity,
           "Seen data": self.dataset_seen,
           "Reward Expl": self.rewards_expl}

    def close(self):
        """Clean up resources."""
        pass


In [ ]:
import torch.optim as optim

from dqn_model import DQN, StateActionQNetwork
from dqn_learn import OptimizerSpec, dqn_learing
from utils.gym import get_env, get_wrapper_by_name
from utils.schedule import LinearSchedule

BATCH_SIZE = 128
GAMMA = 0.99
REPLAY_BUFFER_SIZE = 1000000
LEARNING_STARTS = 50000
LEARNING_FREQ = 4
FRAME_HISTORY_LEN = 16
TARGER_UPDATE_FREQ = 1000
LEARNING_RATE = 0.001 
ALPHA = 0.95
EPS = 0.01
N_STEP = 50

In [ ]:
def main(env, env_test, num_timesteps):

    def stopping_criterion(env):
        # notice that here t is the number of steps of the wrapped env,
        # which is different from the number of steps in the underlying env
        return env.done #env.current_step >= num_timesteps 
    
    optimizer_spec = OptimizerSpec(
        constructor=optim.RMSprop,
        kwargs=dict(lr=LEARNING_RATE, alpha=ALPHA, eps=EPS),
    )

    exploration_schedule = LinearSchedule(1000000, 0.1, initial_p=0.5)

    dqn_learing(
        env=env,
        env_test=env_test,
        q_func=StateActionQNetwork,
        optimizer_spec=optimizer_spec,
        exploration=exploration_schedule,
        stopping_criterion=stopping_criterion,
        replay_buffer_size=REPLAY_BUFFER_SIZE,
        batch_size=BATCH_SIZE,
        gamma=GAMMA,
        learning_starts=LEARNING_STARTS,
        learning_freq=LEARNING_FREQ,
        frame_history_len=FRAME_HISTORY_LEN,
        target_update_freq=TARGER_UPDATE_FREQ,
        save_file_name="./syntab/expl_%dsteps"%N_STEP
    )

In [ ]:
env = AuditingEnv(state_shape=(n_classes*n_classes,), expl_shape=(n_classes*n_features,), action_list_length=len(X_test[:1000]), target_list=target_list, dataset=y_proba[:1000], explanation=expl, dataset_label=y_test[:1000], max_steps=1000)
env_test = AuditingEnv(state_shape=(n_classes*n_classes,), expl_shape=(n_classes*n_features,), action_list_length=len(X4test), target_list=target4test_list, dataset=y4test_proba, explanation=expl_test, dataset_label=y4test, max_steps=N_STEP, test_env=True, index4test_list_expl=index4test_list_expl)
env.reset()
env_test.reset()

main(env, env_test, 100000)